# C12-classical-models — Practice p19 — Solution


Each outer seed creates one pinned bootstrap sample. The single tree and 25-tree bagger both fit only that resample; prediction matrices retain the across-resample evidence.


In [ ]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier

rng_p19 = np.random.default_rng(20260804)
X_train_p19 = rng_p19.normal(size=(120, 4)).astype(np.float64)
y_train_p19 = (X_train_p19[:,0] * X_train_p19[:,1] +
                0.3 * X_train_p19[:,2] > 0).astype(np.int64)
X_validation_p19 = rng_p19.normal(size=(60, 4)).astype(np.float64)
y_validation_p19 = (X_validation_p19[:,0] * X_validation_p19[:,1] +
                     0.3 * X_validation_p19[:,2] > 0).astype(np.int64)


def tree_bag_variance_audit(X_train, y_train, X_validation, y_validation):
    tree_models = []
    bagging_models = []
    tree_predictions = []
    bag_predictions = []
    for offset in range(12):
        seed = 20260804 + offset
        indices = np.random.default_rng(seed).integers(0, X_train.shape[0], size=X_train.shape[0])
        tree = DecisionTreeClassifier(criterion="gini", splitter="best", max_depth=None,
            min_samples_split=2, min_samples_leaf=1, min_weight_fraction_leaf=0.0,
            max_features=None, random_state=seed, max_leaf_nodes=None,
            min_impurity_decrease=0.0, class_weight=None, ccp_alpha=0.0)
        base_tree = DecisionTreeClassifier(criterion="gini", splitter="best", max_depth=None,
            min_samples_split=2, min_samples_leaf=1, min_weight_fraction_leaf=0.0,
            max_features=None, random_state=seed, max_leaf_nodes=None,
            min_impurity_decrease=0.0, class_weight=None, ccp_alpha=0.0)
        bag = BaggingClassifier(estimator=base_tree, n_estimators=25, max_samples=1.0,
            max_features=1.0, bootstrap=True, bootstrap_features=False, oob_score=False,
            warm_start=False, n_jobs=1, random_state=seed)
        tree.fit(X_train[indices], y_train[indices])
        bag.fit(X_train[indices], y_train[indices])
        tree_models.append(tree)
        bagging_models.append(bag)
        tree_predictions.append(tree.predict(X_validation))
        bag_predictions.append(bag.predict(X_validation))
    tree_matrix = np.asarray(tree_predictions, dtype=np.int64)
    bag_matrix = np.asarray(bag_predictions, dtype=np.int64)
    tree_accuracy = np.mean(tree_matrix == y_validation, axis=1)
    bag_accuracy = np.mean(bag_matrix == y_validation, axis=1)
    def summaries(matrix):
        p = matrix.mean(axis=0)
        variance = p * (1.0-p)
        pairwise = 2.0 * variance * matrix.shape[0] / (matrix.shape[0]-1)
        return variance, pairwise
    tree_variance, tree_disagreement = summaries(tree_matrix)
    bag_variance, bag_disagreement = summaries(bag_matrix)
    return {"tree_models": tuple(tree_models), "bagging_models": tuple(bagging_models),
            "tree_predictions": tree_matrix, "bagging_predictions": bag_matrix,
            "tree_accuracies": tree_accuracy, "bagging_accuracies": bag_accuracy,
            "tree_disagreement": tree_disagreement,
            "bagging_disagreement": bag_disagreement,
            "tree_mean_accuracy": float(tree_accuracy.mean()),
            "bagging_mean_accuracy": float(bag_accuracy.mean()),
            "tree_mean_prediction_variance": float(tree_variance.mean()),
            "bagging_mean_prediction_variance": float(bag_variance.mean())}


audit_p19 = tree_bag_variance_audit(
    X_train_p19, y_train_p19, X_validation_p19, y_validation_p19
)
diagnosis_p19 = '''The retained matrices show that bagging reduces mean rowwise prediction variance and pairwise disagreement across the twelve outer resamples. Its mean validation accuracy is also higher here, but accuracy is a separate bias/generalization statistic: variance reduction alone does not guarantee an accuracy increase.'''


### Answer check


In [ ]:
ATOL = 1e-12
RTOL = 1e-10
assert audit_p19["tree_predictions"].shape == (12,60)
assert audit_p19["bagging_predictions"].shape == (12,60)
assert set(audit_p19) == {"tree_models","bagging_models","tree_predictions","bagging_predictions","tree_accuracies","bagging_accuracies","tree_disagreement","bagging_disagreement","tree_mean_accuracy","bagging_mean_accuracy","tree_mean_prediction_variance","bagging_mean_prediction_variance"}
assert np.isclose(audit_p19["tree_mean_accuracy"], 0.6722222222222224, atol=ATOL, rtol=RTOL)
assert np.isclose(audit_p19["bagging_mean_accuracy"], 0.7541666666666665, atol=ATOL, rtol=RTOL)
assert np.isclose(audit_p19["tree_mean_prediction_variance"], 0.15833333333333333, atol=ATOL, rtol=RTOL)
assert np.isclose(audit_p19["bagging_mean_prediction_variance"], 0.12025462962962966, atol=ATOL, rtol=RTOL)
assert audit_p19["bagging_mean_prediction_variance"] < audit_p19["tree_mean_prediction_variance"]
assert isinstance(diagnosis_p19, str) and "variance" in diagnosis_p19
